#Preparazione

In [1]:
#librerie di base
import pandas as pd
import numpy as np

In [2]:
#libreria per l'analisi dei dati
import statsmodels.api as sm
#FFNN
from sklearn.neural_network import MLPClassifier

In [3]:
#libreria di supporto per l'analisi

# per lo scaling delle variabili numeriche
from sklearn.preprocessing import StandardScaler
# per eseguire il trainin e il test e la valutazione del modello
from sklearn.model_selection import train_test_split as envtest
#PER LA VALUTAZIONE
from sklearn.metrics import accuracy_score

In [4]:
# =============================================================
# =============================================================
#
# importo le funzioni personali creati per questa fase


#aggiunto quando ho spostato i file in una cartella a parte
import sys
sys.path.append("./Moduli")

import Moduli.Modulo_Common as pd_common
import Moduli.Modulo_Fase_Analisi as pd_analisi

##recupero dati

In [5]:
df_incidenti = pd.read_csv( pd_common.GetFolderAnalisi() + "//Incidenti_flat.csv")
# DA GOOGLE NOTEBOOK
# df_incidenti = pd.read_csv(".//Analisi//Incidenti_flat.csv")

In [6]:
df_incidenti.columns

Index(['Unnamed: 0', 'TIME_PERIOD', 'COD_RIP', 'DEN_RIP', 'COD_REG', 'DEN_REG',
       'COD_UTS', 'DEN_UTS', 'PRO_COM', 'COMUNE', 'AREA_KMQ', 'POP_RES',
       'POP_AL_KMQ', 'RESULT', 'RESULT_DESC', 'OBS_VALUE', 'TIPO_DATO',
       'TIPO_DATO_DSC'],
      dtype='object')

In [7]:
df_incidenti = df_incidenti.drop('Unnamed: 0', axis=1)

In [8]:
df_incidenti.head(5)

,index,TIME_PERIOD,COD_RIP,DEN_RIP,COD_REG,DEN_REG,COD_UTS,DEN_UTS,PRO_COM,COMUNE,AREA_KMQ,POP_RES,POP_AL_KMQ,RESULT,RESULT_DESC,OBS_VALUE,TIPO_DATO,TIPO_DATO_DSC
0,0,2010,1,Nord-ovest,1,Piemonte,1,Torino,1001,Agliè,13.1462,2622,199.449271,F,Feriti,14,S,Storico
1,1,2011,1,Nord-ovest,1,Piemonte,1,Torino,1001,Agliè,13.1462,2615,198.916797,F,Feriti,6,S,Storico
2,2,2012,1,Nord-ovest,1,Piemonte,1,Torino,1001,Agliè,13.1462,2657,202.111637,F,Feriti,8,S,Storico
3,3,2013,1,Nord-ovest,1,Piemonte,1,Torino,1001,Agliè,13.1462,2722,207.056031,F,Feriti,5,S,Storico
4,4,2014,1,Nord-ovest,1,Piemonte,1,Torino,1001,Agliè,13.1462,2748,209.033789,F,Feriti,11,S,Storico


#preparazione ambiente per analisi

In [15]:
#feature : variabili indipendenti:  'TIME_PERIOD','PRO_COM','POP_AL_KMQ', 'RESULT'
#   NB le altre sono relazionate tra loro per cui le derivo semplicemente sapendo una di quelle considerate come feature
#   NB COME FATTO IN SCRIPT PRECEDENTE LE VARIABILI IND. POP_RES E AREA_KMQ SONO STATE RIDOTTE A UNA SOLA POP_AL_KMQ CHE RACCOGLIE AMBEDUE LE INFORMAZIONI
#     RIDUCENDO GLI EFFETTI NEGATIVI DI INSERIRE PIù FEATURE E DOVE CORREGGERE IL RELATIVO EFFETTO - ES R QUADRO ADJUSTMENT
# variabile target: variabile dipendente: 'OBS_VALUE'
# rimosso  perchè basta POP_AL_KMQ
# RIMOSSO 'PRO_COM' COME DA ANALISI FATTA PRECEDENTEMENTE
df_analisi = df_incidenti[['TIME_PERIOD','POP_AL_KMQ', 'RESULT','OBS_VALUE']]

In [9]:
df_analisi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123978 entries, 0 to 123977
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   TIME_PERIOD  123978 non-null  int64  
 1   POP_AL_KMQ   123978 non-null  float64
 2   RESULT       123978 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 2.8+ MB


In [16]:
df_analisi['RESULT'] = df_analisi['RESULT'].astype('string')

C:\Users\pietr\AppData\Local\Temp\ipykernel_17576\2194147801.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analisi['RESULT'] = df_analisi['RESULT'].astype('string')


In [17]:
df_analisi['MORTI']=0

C:\Users\pietr\AppData\Local\Temp\ipykernel_17576\1966957988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analisi['MORTI']=0


In [18]:
maschera_filtro= df_analisi['RESULT']=='M'
df_analisi.loc[maschera_filtro,'MORTI']=1

In [19]:

nr_morti= df_analisi[maschera_filtro]['MORTI'].agg('size')

In [14]:
df_analisi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123978 entries, 0 to 123977
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   TIME_PERIOD  123978 non-null  int64  
 1   POP_AL_KMQ   123978 non-null  float64
 2   RESULT       123978 non-null  string 
 3   MORTI        123978 non-null  int64  
dtypes: float64(1), int64(2), string(1)
memory usage: 3.8 MB


#definizione dati di analisi

In [20]:
# definizione delle variabile dipendente - target
Y_Target = df_analisi['OBS_VALUE']

In [21]:
# definizione delle feature - variabili indipendenti
# rimosso 'PRO_COM', derivato da POP_AL_KMQ
X_Feature = df_analisi[['TIME_PERIOD','POP_AL_KMQ', 'MORTI']]

##training test


In [22]:
# DEFINISCO GLI AMBIENTI DI TEST E TRAINING
X_train, X_test, Y_train, Y_test = envtest(X_Feature, Y_Target, test_size=0.3,random_state=2020)

# riporto tutti i valori alla medesima scala
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
df_forecast = pd.read_csv( pd_common.GetFolderAnalisi() + "//Forecast_Da_Calcolare.csv")

In [28]:
X_Feature_Forecast = df_forecast[['TIME_PERIOD','POP_AL_KMQ', 'MORTI']]
Y_Target_Forecast = df_forecast['OBS_VALUE']

In [32]:
scaler2 = StandardScaler()
X_Feature_Forecast_scaled = scaler2.fit_transform(X_Feature_Forecast)

#SCELTE COMUNI

In [23]:
FUNZIONE_ATTIVAZIONE = 'logistic' #è LA FUNZIONE SIGMOIDE
#FUNZIONE_ATTIVAZIONE = 'relu' #è LA FUNZIONE Relu...
nr_iterazioni =1000

In [33]:
#ritorna: target predetto, accuracy_modello
def NN_Training_Evaluate( modello, nomefile):
  #allenon il modello sui dati di training
  modello.fit(X_train_scaled, Y_train)
  # eseguo le previsioni sui dati supervisionati
  valori_target_test = modello.predict(X_test_scaled)

  valori_target_forecast = modello.predict(X_Feature_Forecast_scaled)

  # calcolo l'accuratezza sui dati mai visti dal modello ma supervisionati
  accuracy_modello = accuracy_score(Y_test, valori_target_test)
  
  pd_common.SalvaArray(valori_target_test, nomefile,False)
  pd_common.SalvaArray(valori_target_forecast, nomefile+"_Forecast",False)

  return valori_target_test, accuracy_modello, valori_target_forecast

In [21]:
#target, accuracy = NN_Training_Evaluate('prova')
#print(f'target: {target}')
#print(f'accuracy: {accuracy}')

In [34]:
#ritorna: target predetto, accuracy_modello
def Read_Performance_Modello(accuratezza_modello):
  print(f"Neural Network - Test Accuracy: {accuratezza_modello * 100:.1f}%")
  #print(f"\nImprovement over Logistic Regression: +{(accuracy_nn - accuracy_lr) * 100:.1f} percentage points")
  #print(f"\nNetwork architecture: {[X_train_scaled.shape[1]]} input → {list(mlp.hidden_layer_sizes)} hidden → [1] output")
  #print(f"Number of training epochs used: {mlp.n_iter_}")

#CASO NN 1: 1 LAYER HIDDEN INTERNO CON 30 NEURONI



#SCELTA MODELLO

HO SCELTO IL MODELLO CON LA MAGGIORE ACCURATEZZA E IL MINOR NUMERO DI HYPER  PARAMETRI ARCHITETTURALI PER RIDURRE IL RISCHIO DI OVERFITTING.

LI HO COMMENTATI GLI ALTRI PER I TEMPI LUNGHI DI ESECUZIONE

INOLTRE PER NON PERDERE I VALORI PREVISIONALI LI HO SALVATI CON IL NOME DI tARGE_NN_...

In [23]:
#CASO NN 1: 1 LAYER HIDDEN INTERNO CON 30 NEURONI
mlp_1 = MLPClassifier(
        hidden_layer_sizes=(30),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=421
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m1, accuratezza_m1 =  NN_Training_Evaluate(mlp_1)
# Read_Performance_Modello(accuratezza_m1)

#CASO NN 2 PARI: 2 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 2 PARI: 2 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 15 NEURONI
#                   2 LAYER INTERNO CON 15 NEURONI
#CASO NN 1: 1 LAYER HIDDEN INTERNO CON 30 NEURONI
mlp_2_P = MLPClassifier(
        hidden_layer_sizes=(15, 15),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=422
      )

Target_prev_m2p, accuratezza_m2p, Target_forecast_m2p =  NN_Training_Evaluate(mlp_2_P, "Target_nn_2P")
Read_Performance_Modello(accuratezza_m2p)

      path completo file:  c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P.csv
         Il file 'c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P.csv' esiste, per cui lo rimuovo
      Il file salvato c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P.csv
      path completo file:  c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P_Forecast.csv
      Il file 'c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P_Forecast.csv' non esiste
      Il file salvato c:\progetto_dataanalyst\progettofinale\Analisi/Target_nn_2P_Forecast.csv


ValueError: too many values to unpack (expected 2)

#CASO NN 2 TOP-DOWN: 2 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 2 TOP-DOWN: 2 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 20 NEURONI
#                   2 LAYER INTERNO CON 10 NEURONI
mlp_2_TD = MLPClassifier(
        hidden_layer_sizes=(20, 10),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=423
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m2td, accuratezza_m2td =  NN_Training_Evaluate(mlp_2_TD)
# Read_Performance_Modello(accuratezza_m2td)

#CASO NN 2 DOWN-TOP: 2 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 2 DOWN-TOP: 2 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 10 NEURONI
#                   2 LAYER INTERNO CON 20 NEURONI
mlp_2_DT = MLPClassifier(
        hidden_layer_sizes=(10, 20),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=424
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m2dt, accuratezza_m2dt =  NN_Training_Evaluate(mlp_2_DT)
# Read_Performance_Modello(accuratezza_m2dt)

#CASO NN 3 PARI: 3 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 3 PARI: 3 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 10 NEURONI
#                   2 LAYER INTERNO CON 10 NEURONI
#                   3 LAYER INTERNO CON 10 NEURONI
mlp_3_P = MLPClassifier(
        hidden_layer_sizes=(10, 10, 10),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=425
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m3p, accuratezza_m3p =  NN_Training_Evaluate(mlp_3_P)
# Read_Performance_Modello(accuratezza_m3p)

#CASO NN 3 TOP-DOWN: 3 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 3 TOP-DOWN: 3 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 15 NEURONI
#                   2 LAYER INTERNO CON 10 NEURONI
#                   3 LAYER INTERNO CON 5 NEURONI
mlp_3_TD = MLPClassifier(
        hidden_layer_sizes=(15, 10, 5),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=426
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m3td, accuratezza_m3td =  NN_Training_Evaluate(mlp_3_TD)
# Read_Performance_Modello(accuratezza_m3td)

#CASO NN 3 DOWN-TOP: 3 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 3 DOWN-TOP: 3 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 5 NEURONI
#                   2 LAYER INTERNO CON 10 NEURONI
#                   3 LAYER INTERNO CON 15 NEURONI
mlp_3_DT = MLPClassifier(
        hidden_layer_sizes=(5, 10, 15),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=427
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m3dt, accuratezza_m3dt =  NN_Training_Evaluate(mlp_3_DT)
# Read_Performance_Modello(accuratezza_m3dt)

CASO NN 4 pari: 4 LAYER HIDDEN INTERNO

In [ ]:
#CASO NN 4 pari: 4 LAYER HIDDEN INTERNO
#                   1 LAYER INTERNO CON 9 NEURONI
#                   2 LAYER INTERNO CON 7 NEURONI
#                   3 LAYER INTERNO CON 7 NEURONI
#                   4 LAYER INTERNO CON 7 NEURONI
mlp_4_P = MLPClassifier(
        hidden_layer_sizes=(9, 7, 7,7),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=427
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m4p, accuratezza_m4p =  NN_Training_Evaluate(mlp_4_P)
# Read_Performance_Modello(accuratezza_m4p)

In [ ]:
#CASO NN 6 pari: 6 LAYER HIDDEN INTERNO
#                   da 1 a 6 LAYER INTERNO CON 5 NEURONI
mlp_6_P = MLPClassifier(
        hidden_layer_sizes=(5, 5, 5, 5, 5, 5),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=427
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m6p, accuratezza_m6p =  NN_Training_Evaluate(mlp_6_P)
# Read_Performance_Modello(accuratezza_m6p)

In [ ]:
#CASO NN 10 pari: 10 LAYER HIDDEN INTERNO
#                   1-10 LAYER INTERNO CON 3 NEURONI
mlp_10_P = MLPClassifier(
        hidden_layer_sizes=(3, 3, 3, 3, 3, 3, 3 ,3 ,3,3),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=427
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m10p, accuratezza_m10p =  NN_Training_Evaluate(mlp_10_P)
# Read_Performance_Modello(accuratezza_m10p)

In [ ]:
#CASO NN 10 pari: 10 LAYER HIDDEN INTERNO
#                   1-10 LAYER INTERNO CON 3 NEURONI
mlp_100_P = MLPClassifier(
        hidden_layer_sizes=(10, 10, 10, 10, 10, 10, 10 ,10 ,10,10),   # two hidden layers: 16 neurons, then 8
        activation=FUNZIONE_ATTIVAZIONE,             # ReLU activation in hidden layers
        solver='adam',                 # Adam optimizer (advanced gradient descent)
        max_iter=nr_iterazioni,                 # maximum number of training epochs
        random_state=427
      )

#disabilito dopo le prove dei hyper parametri
# Target_prev_m100p, accuratezza_m100p =  NN_Training_Evaluate(mlp_100_P)
# Read_Performance_Modello(accuratezza_m100p)

 #CALCOLO FORECAST 2027

In [ ]:
#df_forecast = pd.read_csv( ".//Analisi//Forecast_Da_Calcolare.csv")
df_forecast = pd.read_csv( pd_common.GetFolderAnalisi() + "//Forecast_Da_Calcolare.csv")